# FIG02: Single-dataset example of the Figure 3 expected-vs-observed analysis.

```
FIG02: Single-dataset example of the Figure 3 expected-vs-observed analysis.

An LFQbench-style view of ONE dataset (site 42 / Talus Orbitrap Exploris 480, the sPRG
stability/homogeneity DIA run): per-peptide observed log2 fold-change vs. log2 abundance
(scatter) beside per-species boxplots, faceted by the three sample comparisons
(A/B, A/C, B/C), with dashed design-expected lines. Methodologically identical to Fig 3,
shown for a single submission as the worked "what good looks like" example.

INPUT: the DIA-NN `report.pr_matrix.tsv` (precursor-level; collapsed to peptide by summing
across charge states). COLS map the A/B/C run columns (DIA-NN names them by full mzML path).

SPECIES ASSIGNMENT: a single run carries no species column, so peptides are assigned by an
in-silico tryptic digest of the combined FASTA, keeping only peptides UNIQUE to one
organism (matching Fig 3's per-species logic). The digest -> peptide/species map is cached
to data/fig2_peptide_species.csv so re-runs are fast.

Colors unified with the other figures: Human=orange, Cow=green, Trout=blue.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA42 = r"D:/2022 Multi-Species Standard Study/42"
INPUT = os.path.join(DATA42, "DIANN_out", "report.pr_matrix.tsv")   # DIA-NN precursor matrix
FASTA = r"D:/2022 Multi-Species Standard Study/fasta/LakeTrout-Human-Cow-Contaminants-sPRG2022.fasta"
PEPTIDE_COL = "Stripped.Sequence"
# columns holding the A/B/C ratio runs (DIA-NN uses the full mzML path as the column name)
COLS = {"A": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_A.mzML",
        "B": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_B.mzML",
        "C": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_C.mzML"}

OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

SPECIES_ORDER = ["Bovine", "Human", "Trout"]
COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}  # human orange, cow green, trout blue
COMPARISONS = ["A/B", "A/C", "B/C"]
MIX = {"A": {"Trout": 50, "Human": 45, "Bovine": 5},
       "B": {"Trout": 50, "Human": 20, "Bovine": 30},
       "C": {"Trout": 50, "Human": 3,  "Bovine": 47}}
EXPECTED = {c: {sp: np.log2(MIX[c[0]][sp] / MIX[c[-1]][sp]) for sp in SPECIES_ORDER}
            for c in COMPARISONS}
ORGBIT = {"Human": 1, "Bovine": 2, "Trout": 4}
INV = {1: "Human", 2: "Bovine", 4: "Trout"}

In [2]:
def clean_pep(p):
    return re.sub(r"[^A-Z]", "", re.sub(r"\[.*?\]", "", str(p).upper()))

In [3]:
def digest(seq, missed=2, min_len=6, max_len=50):
    """Tryptic peptides: cut after K/R not before P, up to `missed` missed cleavages."""
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps

In [4]:
def fasta_org(header):
    tok = header.split()[0]
    if "Cont_" in tok:
        return None          # contaminant -- not part of the 3-proteome design
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"           # trout is UniProt (_SALNM) in the study FASTA

In [5]:
def assign_species(targets):
    """Map each observed (cleaned) peptide to its organism; keep only unique ones.
    Cached to data/fig2_peptide_species.csv."""
    cache = os.path.join(DATA, "fig2_peptide_species.csv")
    if os.path.exists(cache):
        cached = pd.read_csv(cache)
        if set(targets).issubset(set(cached["peptide"].astype(str))):
            return cached                     # cache covers this input; reuse
        print("  cached species map does not cover current peptides -> rebuilding")
    member = {p: 0 for p in targets}

    def flush(header, seq_parts):
        if not header:
            return
        org = fasta_org(header)
        if org is None:                     # skip contaminants
            return
        bit = ORGBIT[org]
        for p in digest("".join(seq_parts)):
            if p in member:
                member[p] |= bit

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)

    rows = [(p, INV.get(b, "SHARED" if b else "NA"), 1 if b in INV else 0)
            for p, b in member.items()]
    df = pd.DataFrame(rows, columns=["peptide", "species", "unique"])
    df.to_csv(cache, index=False)
    return df

In [6]:
def load():
    q = pd.read_csv(INPUT, sep="\t", engine="python", on_bad_lines="skip")
    q.columns = [str(c).strip() for c in q.columns]
    q["peptide"] = q[PEPTIDE_COL].map(clean_pep)
    for k, col in COLS.items():
        q[k] = pd.to_numeric(q[col], errors="coerce")
    # DIA-NN pr_matrix is precursor-level -> collapse to peptide by summing across charges
    q = q.groupby("peptide", as_index=False)[["A", "B", "C"]].sum(min_count=1)
    q = q[(q["A"] > 0) & (q["B"] > 0) & (q["C"] > 0)]

    spec = assign_species(set(q["peptide"]))
    uniq = spec[spec["unique"] == 1][["peptide", "species"]]
    m = q.merge(uniq, on="peptide", how="inner")

    for k in ("A", "B", "C"):
        m["l" + k] = np.log2(m[k])
    recs = []
    for comp in COMPARISONS:
        n, d = comp[0], comp[-1]
        s = m[m["lB"] > 5].copy()          # abundance filter on sample B (matches original)
        recs.append(pd.DataFrame({"species": s["species"], "lB": s["lB"],
                                  "comparison": comp, "log2fc": s["l" + n] - s["l" + d]}))
    long = pd.concat(recs, ignore_index=True).dropna(subset=["lB", "log2fc"])
    return m, long

In [7]:
def figure(long, out_png):
    ylo, yhi = np.nanpercentile(long["log2fc"], [0.5, 99.5])
    xlo, xhi = np.nanpercentile(long["lB"], [0.5, 99.5])
    fig = plt.figure(figsize=(10.5, 9.5))
    gs = fig.add_gridspec(3, 2, width_ratios=[2.5, 1], hspace=0.30, wspace=0.06)
    for r, comp in enumerate(COMPARISONS):
        sc = fig.add_subplot(gs[r, 0])
        bx = fig.add_subplot(gs[r, 1], sharey=sc)
        sub = long[long["comparison"] == comp]
        for sp in SPECIES_ORDER:
            ss = sub[sub["species"] == sp]
            sc.scatter(ss["lB"], ss["log2fc"], s=5, alpha=0.30, color=COLORS[sp], linewidth=0)
            sc.axhline(EXPECTED[comp][sp], ls="--", lw=1.3, color=COLORS[sp], zorder=4)
            if len(ss):                     # observed median ratio annotation
                med = np.median(ss["log2fc"])
                sc.text(xhi + 0.3, med, f"{2**med:.2f}", color=COLORS[sp],
                        va="center", ha="left", fontsize=8, fontweight="bold")
        sc.set_xlim(xlo - 0.5, xhi + 1.6)
        sc.set_ylim(ylo - 0.6, yhi + 0.6)
        sc.set_ylabel(f"{comp}\nobserved log2 fold-change")
        sc.grid(True, lw=0.3)
        if r == len(COMPARISONS) - 1:
            sc.set_xlabel("log2 abundance (sample B)")

        data = [sub[sub["species"] == sp]["log2fc"].values for sp in SPECIES_ORDER]
        bp = bx.boxplot(data, tick_labels=SPECIES_ORDER, widths=0.6,
                        showfliers=False, patch_artist=True)
        for patch, sp in zip(bp["boxes"], SPECIES_ORDER):
            patch.set_facecolor(COLORS[sp]); patch.set_alpha(0.6)
        for med in bp["medians"]:
            med.set_color("black")
        for sp in SPECIES_ORDER:
            bx.axhline(EXPECTED[comp][sp], ls="--", lw=1.0, color=COLORS[sp], zorder=1)
        bx.grid(True, axis="y", lw=0.3)
        plt.setp(bx.get_yticklabels(), visible=False)
        plt.setp(bx.get_xticklabels(), fontsize=8, rotation=30, ha="right")

    handles = [Line2D([0], [0], marker="o", ls="", color=COLORS[s], label=s) for s in SPECIES_ORDER]
    handles += [Line2D([0], [0], ls="--", color="0.3", label="expected log2 ratio")]
    fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.99))
    fig.suptitle("FIG02  Single-dataset example (site 42) — observed vs. expected", y=1.0, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_png.replace(".png", ".pdf"))
    plt.close(fig)
    print(f"  wrote {out_png}")

In [8]:
# ---- generate figure ----
peptides, long = load()
long.to_csv(os.path.join(DATA, "fig2_single_dataset_long.csv"), index=False)
n_uniq = peptides["species"].value_counts().to_dict()
print(f"{len(peptides)} unique-peptide quant rows; per species: {n_uniq}")
figure(long, os.path.join(OUTPUT, "FIG02_single_dataset_example.png"))

6603 unique-peptide quant rows; per species: {'Trout': 2712, 'Bovine': 2411, 'Human': 1480}


C:\Users\linds\AppData\Local\Temp\ipykernel_9144\1001093685.py:42: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=(0, 0, 1, 0.96))


  wrote output\FIG02_single_dataset_example.png
